# Feature Engineering

This notebook derives spatial, geometric, and contextual features used by the machine-learning classification stage. 

**Input**
- Building data after Stage 1 and Stage 2 classification.
- Cleaned contextual OpenStreetMap layers, including land use, POIs, roads, waterways, and railways.

**Output**
- Feature-engineered building data containing the predictors required for machine-learning classification.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import get_coordinates
from scipy.spatial import cKDTree
from shapely.strtree import STRtree
import gc
import os



ROOT_DIR = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR = os.path.join(ROOT_DIR, 'data')

BUILDING_FILE = os.path.join(DATA_DIR, 'processed', 'germany_buildings_classified_stage2.parquet')
LANDUSE_FILE = os.path.join(DATA_DIR, 'processed', 'germany_landuse_mapped.parquet')
POI_FILE = os.path.join(DATA_DIR, 'processed', 'germany_pois_mapped.parquet')
ROAD_FILE = os.path.join(DATA_DIR, 'processed', 'germany_roads_mapped.parquet')
WATERWAY_FILE = os.path.join(DATA_DIR, 'processed', 'germany_waterways_mapped.parquet')
RAILWAY_FILE  = os.path.join(DATA_DIR, 'processed', 'germany_railways_mapped.parquet')
OUTPUT_FILE = os.path.join(DATA_DIR, 'processed', 'germany_buildings_feature_engineered.parquet')



In [2]:
print("Loading data...")
gdf_bldg   = gpd.read_parquet(BUILDING_FILE)
gdf_landuse = gpd.read_parquet(LANDUSE_FILE)
gdf_poi    = gpd.read_parquet(POI_FILE)
gdf_roads  = gpd.read_parquet(ROAD_FILE)

print(f"  Buildings : {len(gdf_bldg):,}  | CRS: {gdf_bldg.crs.to_string()}")
print(f"  Landuse   : {len(gdf_landuse):,}")
print(f"  POIs      : {len(gdf_poi):,}")
print(f"  Roads     : {len(gdf_roads):,}")

Loading data...
  Buildings : 38,802,372  | CRS: EPSG:4326
  Landuse   : 3,294,732
  POIs      : 879,768
  Roads     : 8,274,952


In [3]:
print("Column check:")
print(f"  gdf_bldg    : {gdf_bldg.columns.tolist()}")
print(f"  gdf_landuse : {gdf_landuse.columns.tolist()}")
print(f"  gdf_poi     : {gdf_poi.columns.tolist()}")
print(f"  gdf_roads   : {gdf_roads.columns.tolist()}")

print("\ngdf_landuse zone_l1:")
print(gdf_landuse['zone_l1'].value_counts())

print("\ngdf_poi poi_l1:")
print(gdf_poi['poi_l1'].value_counts())

print("\ngdf_roads road_category:")
print(gdf_roads['road_category'].value_counts())

print("\ngdf_bldg geometry types:")
print(gdf_bldg.geometry.geom_type.value_counts())

Column check:
  gdf_bldg    : ['id', 'geometry', 'building', 'tag_l1', 'tag_l2', 'is_abandoned', 'tag_is_mixed', 'tag_source', 'tag_used', 'all_candidates', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'name', 'amenity', 'building:use', 'craft', 'office', 'shop', 'tags', 'building:levels', 'address_tier', 'area_sqm', 'perimeter_m', 'compactness', 'building_l1', 'building_l2', 'building_is_mixed', 'buse_l1', 'buse_l2', 'buse_is_mixed', 'amenity_l1', 'amenity_l2', 'shop_l1', 'shop_l2', 'stage1_l1', 'stage1_l2', 'stage1_is_mixed', 'stage1_source', 'raw_label', 'stage2_l1', 'stage2_l2', 'stage2_source', 'landuse_l1', 'landuse_l2', 'landuse_used', 'landuse_osm_id']
  gdf_landuse : ['id', 'geometry', 'landuse', 'zone_l1', 'zone_l2', 'industrial', 'tags', 'zone_area_sqm']
  gdf_poi     : ['id', 'poi_l1', 'poi_l2', 'poi_is_mixed', 'poi_source', 'poi_used', 'all_candidates', 'geometry', 'building', 'amenity', 'shop', 'tourism']
  gdf_roads   : ['id', 'geometry', 'highway', '

In [4]:
# All spatial feature blocks below use these shared projections.

print("Projecting all layers to EPSG:3035 (metric)...")

bldg_proj  = gdf_bldg.to_crs(epsg=3035)
lu_proj    = gdf_landuse.to_crs(epsg=3035)
roads_proj = gdf_roads.to_crs(epsg=3035)
poi_proj   = gdf_poi[gdf_poi['poi_l1'].notna()].to_crs(epsg=3035).copy()
poi_proj['geometry'] = poi_proj.geometry.centroid  # force all POIs to points

# Pre-compute building centroids
bldg_centroids = bldg_proj.geometry.centroid
bldg_coords    = np.column_stack([bldg_centroids.x, bldg_centroids.y])

print(f"  bldg_coords shape : {bldg_coords.shape}")
print("  Ready.")

Projecting all layers to EPSG:3035 (metric)...
  bldg_coords shape : (38802372, 2)
  Ready.


In [5]:
# Building levels
gdf_bldg['has_building_levels'] = pd.to_numeric(
    gdf_bldg['building:levels'], errors='coerce'
).notna().astype('int8')

# Create 'building_levels' as numeric, keeping NaN for missing values
gdf_bldg['building_levels'] = pd.to_numeric(
    gdf_bldg['building:levels'], errors='coerce'
)

gdf_bldg.drop(columns=['building:levels'], inplace=True)

print(f"has_building_levels: {gdf_bldg['has_building_levels'].sum():,} buildings have level data")
print(f"building_levels nulls: {gdf_bldg['building_levels'].isna().sum():,} ({gdf_bldg['building_levels'].isna().mean()*100:.1f}%)")
print("building_levels will stay as NaN — tree models handle this natively, for other models we can impute or add a 'missing' indicator later if needed.")

has_building_levels: 3,656,602 buildings have level data
building_levels nulls: 35,145,770 (90.6%)
building_levels will stay as NaN — tree models handle this natively, for other models we can impute or add a 'missing' indicator later if needed.


In [6]:
# has name
gdf_bldg['has_name'] = gdf_bldg['name'].notna().astype('int8')
print(f"has_name: {gdf_bldg['has_name'].sum():,} buildings have a name")

has_name: 598,000 buildings have a name


### Compute Geometric Features

In [ ]:
def get_mrr_dimensions(geom):
    mrr    = geom.minimum_rotated_rectangle
    coords = np.array(mrr.exterior.coords)
    sides  = np.sqrt(np.diff(coords[:, 0])**2 + np.diff(coords[:, 1])**2)
    return sides.max(), sides.min()

def compute_geometric_features(gdf):
    geom = gdf.geometry

    print('  area...')
    gdf['area']      = geom.area
    print('  perimeter...')
    gdf['perimeter'] = geom.length
    print('  compactness...') 
    # Compactness = 4π * Area / Perimeter², normalized to [0,1]
    # compactness is how closely the shape resembles a circle (1 = perfect circle, 0 = very elongated)
    gdf['compactness'] = ((4 * np.pi * gdf['area']) / (gdf['perimeter'] ** 2)).clip(0, 1)
    print('  convexity...')
    # Convexity = Area / Convex Hull Area, normalized to [0,1]
    # convexity is how closely the shape resembles its convex hull (1 = convex, 0 = very concave)
    gdf['convexity']   = gdf['area'] / geom.convex_hull.area
    print('  num_vertices...')
    # Number of vertices in the polygon (complexity)
    gdf['num_vertices'] = geom.apply(lambda g: get_coordinates(g).shape[0])
    print('  MRR dimensions...')
    # Minimum Rotated Rectangle (MRR) dimensions: long and short sides
    # MMR is the smallest rectangle that can enclose the shape, and its sides are not necessarily aligned with the axes
    # MRR long is the length of the longest side, MRR short is the length of the shortest side of the MRR 
    # Elongation = MRR long / MRR short, how elongated the shape is (1 = square, >1 = more elongated)
    # Rectangularity = Area / MRR Area, how closely the shape fills its MRR (1 = perfect rectangle, 0 = very irregular)
    # Diameter = diagonal of the MRR, a size measure that is less sensitive to shape than area or perimeter
    mrr_dims          = pd.DataFrame(
        geom.apply(get_mrr_dimensions).tolist(),
        index=geom.index,
        columns=['mrr_long', 'mrr_short']
    )
    gdf['mrr_long']  = mrr_dims['mrr_long']
    gdf['mrr_short'] = mrr_dims['mrr_short']
    print('  elongation, rectangularity, diameter...')
    gdf['elongation']     = gdf['mrr_long'] / gdf['mrr_short'].replace(0, np.nan)
    gdf['mrr_area']       = gdf['mrr_long'] * gdf['mrr_short']
    gdf['rectangularity'] = gdf['area'] / gdf['mrr_area'].replace(0, np.nan)
    gdf['diameter']       = np.sqrt(gdf['mrr_long']**2 + gdf['mrr_short']**2)
    gdf.drop(columns=['mrr_area'], inplace=True)
    return gdf


In [8]:
print("Computing geometric features...")
bldg_proj = compute_geometric_features(bldg_proj)

# Copy back to main dataframe
geo_cols = ['area', 'perimeter', 'compactness', 'convexity',
            'num_vertices', 'mrr_long', 'mrr_short',
            'elongation', 'rectangularity', 'diameter']
for col in geo_cols:
    gdf_bldg[col] = bldg_proj[col]

# Drop old columns if they exist from a previous run
gdf_bldg.drop(columns=['area_sqm', 'perimeter_m'], inplace=True, errors='ignore')

print(gdf_bldg[geo_cols].describe().round(2))


Computing geometric features...
  area...
  perimeter...
  compactness...
  convexity...
  num_vertices...
  MRR dimensions...
  elongation, rectangularity, diameter...
              area    perimeter  compactness    convexity  num_vertices  \
count  38802372.00  38802372.00  38802372.00  38802372.00   38802372.00   
mean        161.21        46.74         0.70         0.97          6.71   
std         579.69        36.13         0.10         0.06          3.55   
min           0.01         1.00         0.00         0.01          4.00   
25%          44.93        28.03         0.66         0.97          5.00   
50%          95.63        40.75         0.73         1.00          5.00   
75%         162.04        54.12         0.77         1.00          7.00   
max      480319.44      4564.99         1.00         1.00        585.00   

          mrr_long    mrr_short   elongation  rectangularity     diameter  
count  38802372.00  38802372.00  38802372.00     38802372.00  38802372.00  
mea

### Nearest Building Distance

In [ ]:
# ── Masks & Geometry Arrays ──────────────────────────────────────────────────
geoms       = bldg_proj.geometry.values


final_l1 = gdf_bldg['stage2_l1'].combine_first(gdf_bldg['stage1_l1'])
final_l2 = gdf_bldg['stage2_l2'].combine_first(gdf_bldg['stage1_l2'])

is_filter = final_l1.eq('filter')
is_animal_keeping = final_l2.eq('animal_keeping')

non_filter = ~(is_filter | is_animal_keeping).to_numpy()
real_geoms = geoms[non_filter]

# filter like structures, aviary, sty etc.
is_animal_keeping = gdf_bldg['stage1_l2'] == 'animal_keeping'

non_filter = ~(is_filter | is_animal_keeping).values  # True = "real" building

real_geoms  = geoms[non_filter]                            # subset for the clean tree




# ── Tree 2: REAL buildings only (filtered excluded) ──────────────────────────
# exclusive=True works here because shapely checks geometric equality,
# so non-filtered buildings won't match themselves in this tree either.
print("Building Real-Buildings STRtree (excluding filtered)...")
tree_real = STRtree(real_geoms)

idx_real, dist_real = tree_real.query_nearest(geoms, exclusive=True, return_distance=True)
_, uf_real = np.unique(idx_real[0], return_index=True)
clean_dist_real = dist_real[uf_real]

gdf_bldg['dist_nearest_real_building']   = clean_dist_real.astype(np.float32)
gdf_bldg['is_touching_real_building']    = (clean_dist_real <= 3.0).astype(np.int8)

left_real, right_real = tree_real.query(real_geoms, predicate='touches')
mask_real = left_real != right_real
real_touch_counts = np.bincount(left_real[mask_real], minlength=len(real_geoms)).astype(np.int8)

# Map counts back to the full index (filtered buildings get 0)
full_touch_real = np.zeros(len(geoms), dtype=np.int8)
full_touch_real[non_filter] = real_touch_counts
gdf_bldg['real_touching_count'] = full_touch_real

del tree_real, idx_real, dist_real, clean_dist_real
del left_real, right_real, mask_real, real_touch_counts, full_touch_real
gc.collect()


# ── Audit ────────────────────────────────────────────────────────────────────
print("\n--- Feature Audit ---")
print(f"Within 3m (real only)  : {gdf_bldg['is_touching_real_building'].sum():>10,}")
print(f"Max touching (real)    : {gdf_bldg['real_touching_count'].max():>10}")
print(f"\nReal buildings in tree : {non_filter.sum():>10,}")
print(f"Filtered buildings     : {(~non_filter).sum():>10,}")


--- Feature Audit ---
Within 3m (real only)  : 20,503,722
Max touching (real)    :         26

Real buildings in tree : 28,971,064
Filtered buildings     :  9,831,308


### Distance to Landuse Zones

In [13]:
def dist_to_landuse(bldg_coords, bldg_proj_centroids, lu_proj, zone_type,
                         max_dist=5000, max_segment_m=50):
    zone_gdf = lu_proj[lu_proj['zone_l1'] == zone_type]  

    if zone_gdf.empty:
        print(f"  WARNING: no '{zone_type}' zones found — filling with max_dist ({max_dist}m)")
        return np.full(len(bldg_coords), max_dist, dtype=np.float32)

    # Densify → extract → deduplicate vertices
    dense_geoms = zone_gdf.geometry.segmentize(max_segment_length=max_segment_m)
    coords      = get_coordinates(dense_geoms)
    print(f"  {zone_type}: {len(coords):,} raw vertices")
    coords      = np.unique(coords, axis=0)         
    print(f"  {zone_type}: {len(coords):,} unique vertices in tree")

    tree = cKDTree(coords)
    distances, _ = tree.query(bldg_coords, k=1, distance_upper_bound=max_dist)
    distances     = np.where(distances == np.inf, max_dist, distances).astype(np.float32)

    # Zero out buildings inside zone polygons (vectorised)
    str_tree   = STRtree(zone_gdf.geometry.values)
    result     = str_tree.query(bldg_proj_centroids, predicate='intersects')
    if result.size > 0:
        inside_idx = np.unique(result[0])
        distances[inside_idx] = 0.0
        print(f"  {zone_type}: {len(inside_idx):,} buildings zeroed (inside zone)")

    return distances


In [14]:
for zone in ['residential', 'commercial', 'industrial', 'agricultural']:
    print(f"\nProcessing landuse zone: '{zone}'...")
    gdf_bldg[f'dist_to_{zone}'] = dist_to_landuse(
        bldg_coords         = bldg_coords,
        bldg_proj_centroids = bldg_centroids.values,
        lu_proj             = lu_proj,
        zone_type           = zone,
        max_dist            = 5000,
        max_segment_m       = 50,
    )
    print(gdf_bldg[f'dist_to_{zone}'].describe().round(1).to_string())



Processing landuse zone: 'residential'...
  residential: 13,892,406 raw vertices
  residential: 13,350,686 unique vertices in tree
  residential: 27,107,398 buildings zeroed (inside zone)
count    38802372.0
mean          138.4
std           320.1
min             0.0
25%             0.0
50%             0.0
75%            92.4
max          5000.0

Processing landuse zone: 'commercial'...
  commercial: 1,808,017 raw vertices
  commercial: 1,688,596 unique vertices in tree
  commercial: 730,202 buildings zeroed (inside zone)
count    38802372.0
mean          994.9
std          1088.1
min             0.0
25%           271.1
50%           574.0
75%          1295.7
max          5000.0

Processing landuse zone: 'industrial'...
  industrial: 1,865,457 raw vertices
  industrial: 1,760,974 unique vertices in tree
  industrial: 664,435 buildings zeroed (inside zone)
count    38802372.0
mean         1024.3
std           932.8
min             0.0
25%           385.3
50%           743.2
75%        

In [ ]:
# Zone Membership Flags + Zone Cluster Features

print("Computing zone membership flags and cluster features...")

# Zone cluster features (for industrial subtype discrimination) ──
# How many buildings share the same land use polygon?

if 'landuse_osm_id' in gdf_bldg.columns:
    print("  landuse_osm_id already present — computing cluster stats...")

    # Count buildings per zone polygon
    zone_building_counts = (
        gdf_bldg[gdf_bldg['landuse_osm_id'].notna()]
        .groupby('landuse_osm_id')
        .size()
        .rename('buildings_in_same_zone')
    )

    gdf_bldg['buildings_in_same_zone'] = (
        gdf_bldg['landuse_osm_id']
        .map(zone_building_counts)
        .fillna(0)
        .astype(np.int32)
    )

    # Zone total area — join from landuse
    zone_area_lookup = (
        gdf_landuse[gdf_landuse['zone_area_sqm'].notna()]
        .set_index('id')['zone_area_sqm']
    )
    gdf_bldg['zone_total_area'] = (
        gdf_bldg['landuse_osm_id']
        .map(zone_area_lookup)
        .fillna(0)
        .astype(np.float32)
    )

    # Building's share of its zone (distinguishes one large warehouse vs many factory units)
    gdf_bldg['building_area_fraction'] = np.where(
        gdf_bldg['zone_total_area'] > 0,
        gdf_bldg['area'] / gdf_bldg['zone_total_area'],
        0.0
    ).astype(np.float32)

    # Zone building density (buildings per hectare)
    gdf_bldg['zone_building_density'] = np.where(
        gdf_bldg['zone_total_area'] > 0,
        gdf_bldg['buildings_in_same_zone'] / (gdf_bldg['zone_total_area'] / 10_000),
        0.0
    ).astype(np.float32)

    print(f"  buildings_in_same_zone  — median: {gdf_bldg['buildings_in_same_zone'].median():.0f}")
    print(f"  building_area_fraction  — median: {gdf_bldg['building_area_fraction'].median():.4f}")
    print(f"  zone_building_density   — median: {gdf_bldg['zone_building_density'].median():.2f}")

else:
    print("  WARNING: landuse_osm_id not found — skipping cluster features")
    print("  Re-run 04a_landuse_eda_cleaning.ipynb and ensure it saves landuse_osm_id to the building file")
    for col in ['buildings_in_same_zone', 'zone_total_area', 'building_area_fraction', 'zone_building_density']:
        gdf_bldg[col] = np.float32(0)

# ── A3: Neighbour building size statistics (200m radius) ─────────────
# What are the surrounding buildings like? Large uniform = manufacturing
print("  Computing neighbour area statistics (200m)...")

# Use already-projected building centroids and STRtree
bldg_areas = bldg_proj['area'].values
bldg_geoms_proj = bldg_proj.geometry.values

tree_area = STRtree(bldg_geoms_proj)
# query_nearest is expensive for all buildings; use dwithin instead
q_left, q_right = tree_area.query(bldg_geoms_proj, predicate='dwithin', distance=200)

# Exclude self-matches
mask_self = q_left != q_right
q_left  = q_left[mask_self]
q_right = q_right[mask_self]

neighbour_areas = bldg_areas[q_right]

# Aggregate per building
neighbour_mean = np.zeros(len(gdf_bldg), dtype=np.float32)
neighbour_max  = np.zeros(len(gdf_bldg), dtype=np.float32)
neighbour_std  = np.zeros(len(gdf_bldg), dtype=np.float32)

print("  Aggregating neighbor stats (vectorized)...")

# Put the arrays into a fast pandas DataFrame
df_neighbors = pd.DataFrame({
    'bldg_idx': q_left,
    'area': neighbour_areas
})

# Group by the building index and calculate all stats at once in C-speed
agg_stats = df_neighbors.groupby('bldg_idx')['area'].agg(['mean', 'max', 'std']).fillna(0)

# Assign the results back to the pre-allocated numpy arrays using the indices
neighbour_mean[agg_stats.index] = agg_stats['mean'].values
neighbour_max[agg_stats.index]  = agg_stats['max'].values
neighbour_std[agg_stats.index]  = agg_stats['std'].values

# Add to Geodataframe
gdf_bldg['neighbour_mean_area_200m'] = neighbour_mean
gdf_bldg['neighbour_max_area_200m']  = neighbour_max
gdf_bldg['neighbour_std_area_200m']  = neighbour_std

del tree_area, q_left, q_right, mask_self, neighbour_areas
del df_neighbors, agg_stats, neighbour_mean, neighbour_max, neighbour_std
gc.collect()
print("  Neighbour area features done.")

zone_membership_cols = [
    'buildings_in_same_zone', 'zone_total_area', 'building_area_fraction',
    'zone_building_density',
    'neighbour_mean_area_200m', 'neighbour_max_area_200m', 'neighbour_std_area_200m',
]
print(f"\n {len(zone_membership_cols)} new features.")

Computing zone membership flags and cluster features...
  landuse_osm_id already present — computing cluster stats...
  buildings_in_same_zone  — median: 201
  building_area_fraction  — median: 0.0002
  zone_building_density   — median: 12.51
  Computing neighbour area statistics (200m)...
  Aggregating neighbor stats (vectorized)...
  Neighbour area features done.

 7 new features.


In [16]:
del lu_proj
gc.collect()
print("\nLanduse done. lu_proj freed.")


Landuse done. lu_proj freed.


### POI Density & Distance

In [17]:
POI_FILTERS = {
    'retail':           poi_proj['poi_l2'] == 'retail',
    'office':           poi_proj['poi_l2'] == 'office',
    'food':             poi_proj['poi_l2'] == 'food_drink',
    'other_service': poi_proj['poi_l2'] == 'other_service',  
    'accommodation': poi_proj['poi_l2'] == 'accommodation',
    'healthcare':       poi_proj['poi_l2'].isin(['clinic', 'hospital', 'care_facility']),
    'education':        poi_proj['poi_l2'].isin(['school', 'kindergarten', 'higher_ed']),
    'civic':            poi_proj['poi_l1'] == 'civic',
    'transport':        poi_proj['poi_l1'] == 'transportation'
}

# Tell the script which categories get which math
NEEDS_DENSITY  = [
    'retail', 'office', 'food', 'other_service', 
    'accommodation', 'civic', 'healthcare', 'education'
]
NEEDS_DISTANCE = ['transport', 'education']

for cat, mask in POI_FILTERS.items():
    print(f"\nProcessing POI: '{cat}'...")
    subset = poi_proj[mask]

    if subset.empty:
        print(f"  WARNING: no '{cat}' POIs found")
        if cat in NEEDS_DENSITY:
            for suffix in ['25m', '50m', '100m', '250m']:
                gdf_bldg[f'poi_count_{cat}_{suffix}'] = 0
        if cat in NEEDS_DISTANCE:
            gdf_bldg[f'dist_nearest_{cat}'] = np.float32(5000.0)
        continue

    cat_coords = np.column_stack([subset.geometry.x, subset.geometry.y])
    tree       = cKDTree(cat_coords)

    if cat in NEEDS_DENSITY:
        for radius, suffix in [(25, '25m'), (50, '50m'), (100, '100m'), (250, '250m')]:
            indices = tree.query_ball_point(bldg_coords, r=radius, workers=-1)
            gdf_bldg[f'poi_count_{cat}_{suffix}'] = np.array(
                [len(i) for i in indices], dtype=np.int32
            )
            print(f"  poi_count_{cat}_{suffix}: {gdf_bldg[f'poi_count_{cat}_{suffix}'].sum():,} total hits")

    if cat in NEEDS_DISTANCE:
        dist, _ = tree.query(bldg_coords, k=1, distance_upper_bound=5000)
        gdf_bldg[f'dist_nearest_{cat}'] = np.where(
            dist == np.inf, 5000, dist
        ).astype(np.float32)
        print(f"  dist_nearest_{cat}: median={np.median(gdf_bldg[f'dist_nearest_{cat}']):,.0f}m")

print("\nPOI density/distance done.")




Processing POI: 'retail'...
  poi_count_retail_25m: 956,130 total hits
  poi_count_retail_50m: 3,521,530 total hits
  poi_count_retail_100m: 13,057,099 total hits
  poi_count_retail_250m: 71,609,152 total hits

Processing POI: 'office'...
  poi_count_office_25m: 136,684 total hits
  poi_count_office_50m: 544,614 total hits
  poi_count_office_100m: 2,119,212 total hits
  poi_count_office_250m: 11,931,649 total hits

Processing POI: 'food'...
  poi_count_food_25m: 701,058 total hits
  poi_count_food_50m: 2,600,084 total hits
  poi_count_food_100m: 9,591,085 total hits
  poi_count_food_250m: 51,329,872 total hits

Processing POI: 'other_service'...
  poi_count_other_service_25m: 313,179 total hits
  poi_count_other_service_50m: 1,161,061 total hits
  poi_count_other_service_100m: 4,205,550 total hits
  poi_count_other_service_250m: 22,276,741 total hits

Processing POI: 'accommodation'...
  poi_count_accommodation_25m: 2,901 total hits
  poi_count_accommodation_50m: 9,631 total hits
  po

In [ ]:
# POI RATIO FEATURES
# All ratios use a small epsilon (1e-3) to avoid division by zero.
# Every ratio is clipped to [0, 1] so the model sees a bounded signal.
# We compute at both 100m and 250m so the model can learn at two scales.

EPS = 1e-3

for suffix in ['100m', '250m']:
    retail  = gdf_bldg[f'poi_count_retail_{suffix}']
    food    = gdf_bldg[f'poi_count_food_{suffix}']
    office  = gdf_bldg[f'poi_count_office_{suffix}']
    accom   = gdf_bldg[f'poi_count_accommodation_{suffix}']
    service = gdf_bldg[f'poi_count_other_service_{suffix}']
    
    # Total commercial POIs in radius — denominator for everything below
    total_comm = retail + food + office + accom + service + EPS

    # --- food_drink discrimination ---
    # High value → surrounded by food POIs → likely food building
    # Low value  → surrounded by retail    → likely retail
    gdf_bldg[f'food_share_{suffix}'] = (food / total_comm).clip(0, 1).astype(np.float32)

    # --- retail discrimination ---
    # Pure retail dominance in neighborhood
    gdf_bldg[f'retail_share_{suffix}'] = (retail / total_comm).clip(0, 1).astype(np.float32)

    # --- office discrimination ---
    # High office share → office building rather than street-level commercial
    gdf_bldg[f'office_share_{suffix}'] = (office / total_comm).clip(0, 1).astype(np.float32)

    # --- accommodation discrimination ---
    # Hotels cluster with other hotels; retail and food do not
    gdf_bldg[f'accom_share_{suffix}'] = (accom / total_comm).clip(0, 1).astype(np.float32)

    # --- food vs retail head-to-head ---
    
    gdf_bldg[f'food_vs_retail_{suffix}'] = (
        food / (food + retail + EPS)
    ).clip(0, 1).astype(np.float32)

    # --- office vs retail head-to-head ---
    gdf_bldg[f'office_vs_retail_{suffix}'] = (
        office / (office + retail + EPS)
    ).clip(0, 1).astype(np.float32)

    # --- commercial diversity index ---
    # A pedestrian shopping street has high retail share → low diversity.
    # A mixed urban block has more diversity → may signal food/office zones.
    shares = np.stack([
        (retail  / total_comm).clip(0, 1),
        (food    / total_comm).clip(0, 1),
        (office  / total_comm).clip(0, 1),
        (accom   / total_comm).clip(0, 1),
    ], axis=1)
    herfindahl = (shares ** 2).sum(axis=1)
    gdf_bldg[f'comm_diversity_{suffix}'] = (1 - herfindahl).clip(0, 1).astype(np.float32)

print("\nRatio features done.")
print("New ratio columns:")
ratio_cols = [c for c in gdf_bldg.columns if any(
    c.startswith(p) for p in ['food_share', 'retail_share', 'office_share',
                               'accom_share', 'food_vs_retail', 'office_vs_retail',
                               'comm_diversity']
)]
print(ratio_cols)
print(gdf_bldg[ratio_cols].describe().round(3))


Ratio features done.
New ratio columns:
['food_share_100m', 'retail_share_100m', 'office_share_100m', 'accom_share_100m', 'food_vs_retail_100m', 'office_vs_retail_100m', 'comm_diversity_100m', 'food_share_250m', 'retail_share_250m', 'office_share_250m', 'accom_share_250m', 'food_vs_retail_250m', 'office_vs_retail_250m', 'comm_diversity_250m']
       food_share_100m  retail_share_100m  office_share_100m  \
count     3.880237e+07       3.880237e+07       3.880237e+07   
mean      6.800000e-02       8.200000e-02       1.500000e-02   
std       2.230000e-01       2.440000e-01       9.800000e-02   
min       0.000000e+00       0.000000e+00       0.000000e+00   
25%       0.000000e+00       0.000000e+00       0.000000e+00   
50%       0.000000e+00       0.000000e+00       0.000000e+00   
75%       0.000000e+00       0.000000e+00       0.000000e+00   
max       1.000000e+00       1.000000e+00       1.000000e+00   

       accom_share_100m  food_vs_retail_100m  office_vs_retail_100m  \
count 

In [33]:
print(gdf_bldg[[c for c in gdf_bldg.columns if c.startswith('poi_') or c.startswith('dist_nearest_')]].describe())

       dist_nearest_real_building  poi_count_retail_25m  poi_count_retail_50m  \
count                3.880237e+07          3.880237e+07          3.880237e+07   
mean                 7.244339e+00          2.464102e-02          9.075553e-02   
std                  3.022536e+01          2.259994e-01          6.316572e-01   
min                  0.000000e+00          0.000000e+00          0.000000e+00   
25%                  0.000000e+00          0.000000e+00          0.000000e+00   
50%                  2.429859e+00          0.000000e+00          0.000000e+00   
75%                  7.479296e+00          0.000000e+00          0.000000e+00   
max                  1.160186e+04          2.300000e+01          7.200000e+01   

       poi_count_retail_100m  poi_count_retail_250m  poi_count_office_25m  \
count           3.880237e+07           3.880237e+07          3.880237e+07   
mean            3.365026e-01           1.845484e+00          3.522568e-03   
std             1.835088e+00           

### POI Containment (inside building)

In [20]:
print("Checking POI containment inside buildings...")

poi_str_tree = STRtree(poi_proj.geometry.values)
result       = poi_str_tree.query(bldg_proj.geometry.values, predicate='contains')

# Count of POIs inside each building
counts = np.bincount(result[0], minlength=len(gdf_bldg))
gdf_bldg['poi_count_inside'] = counts.astype(np.int32)

print(f"  Buildings with POI inside : {gdf_bldg['poi_count_inside'].gt(0).sum():,}")

del poi_proj
gc.collect()
print("poi_proj freed.")

Checking POI containment inside buildings...
  Buildings with POI inside : 566,340
poi_proj freed.


### Road Distance & Binary Flags

In [21]:
unique_categories = roads_proj['road_category'].dropna().unique()
print(unique_categories)

roads_by_cat = {
    cat: roads_proj[roads_proj['road_category'] == cat].copy()
    for cat in unique_categories
}

print('\nRoads per category:')
for cat, df in roads_by_cat.items():
    print(f'  {cat:<20}: {len(df):,}')

dist_cols = []
MAX_DIST = 5000.0  # Cap distance at 5km to ignore irrelevant noise

print("\nComputing road distances...")

for cat_name, cat_roads in roads_by_cat.items():
    col = f'dist_nearest_{cat_name}'
    print(f'  {col}...')

    if cat_roads.empty:
        gdf_bldg[col] = np.float32(MAX_DIST)
        dist_cols.append(col)
        continue

    # Segmentize lines into points every 50m for accurate tree matching
    dense_lines = cat_roads.geometry.segmentize(max_segment_length=50)
    
    coords_df = dense_lines.get_coordinates()
    road_coords = np.unique(np.column_stack([coords_df.x, coords_df.y]), axis=0)

    tree = cKDTree(road_coords)
    dist, _ = tree.query(bldg_coords, k=1, distance_upper_bound=MAX_DIST)
    
    # Assign distances 
    gdf_bldg[col] = np.where(dist == np.inf, MAX_DIST, dist).astype(np.float32)
    dist_cols.append(col)

print(f'\nAll {len(dist_cols)} road distance features computed.')

['residential_road' 'secondary_road' 'rural_track' 'major_road'
 'service_road' 'pedestrian_zone']

Roads per category:
  residential_road    : 2,187,012
  secondary_road      : 1,008,301
  rural_track         : 688,928
  major_road          : 426,459
  service_road        : 3,945,173
  pedestrian_zone     : 19,079

Computing road distances...
  dist_nearest_residential_road...
  dist_nearest_secondary_road...
  dist_nearest_rural_track...
  dist_nearest_major_road...
  dist_nearest_service_road...
  dist_nearest_pedestrian_zone...

All 6 road distance features computed.


In [22]:
# List the columns you care about
dist_cols = [
    'dist_nearest_major_road', 
    'dist_nearest_secondary_road', 
    'dist_nearest_residential_road',
    'dist_nearest_rural_track',
    'dist_nearest_service_road',
    'dist_nearest_pedestrian_zone'
]

# Get the median for all columns grouped by building type
full_summary = gdf_bldg.groupby('stage1_l1')[dist_cols].median().round(1)

print("Target-Conditioned Medians (in meters):")
full_summary

Target-Conditioned Medians (in meters):


,dist_nearest_major_road,dist_nearest_secondary_road,dist_nearest_residential_road,dist_nearest_rural_track,dist_nearest_service_road,dist_nearest_pedestrian_zone
stage1_l1,,,,,,
agricultural,1815.599976,227.199997,113.099998,202.399994,47.299999,3314.600098
civic,947.099976,140.699997,38.799999,359.500000,34.200001,1658.099976
commercial,681.799988,112.800003,43.000000,324.899994,26.500000,1462.800049
filter,962.900024,192.100006,23.799999,337.799988,44.900002,1584.800049
industrial,846.299988,197.899994,83.000000,261.600006,31.700001,1918.900024
military,1571.099976,905.200012,864.000000,474.399994,31.500000,3231.600098
residential,937.500000,176.100006,21.000000,374.299988,49.099998,1515.400024
semi_commercial,422.700012,73.000000,24.799999,580.299988,37.099998,1039.000000
transportation,751.400024,142.600006,51.400002,346.399994,25.200001,1435.199951


Thresholds chosen based on the median distance per category

In [23]:
MEANINGFUL_THRESHOLD = {
    'major_road'       : 700,  # commercial
    'secondary_road'   : 115,  # good indicator for commercial
    'residential_road' : 25,   # good indicator for residential
    'service_road'     : 35,   # commercial/industrial buildings directly served by service/industrial access road
    'rural_track'      : 250,   # industrial/agricultural buildings along rural tracks
    'pedestrian_zone'  : 25,   # Kept strict based on physical reality
}

binary_cols = []
print('\nApplying Binary Thresholds...')

for cat_name, threshold in MEANINGFUL_THRESHOLD.items():
    dist_col   = f'dist_nearest_{cat_name}'
    flag_col   = f'near_{cat_name}_{threshold}m'

    if dist_col not in gdf_bldg.columns:
        continue

    gdf_bldg[flag_col] = (gdf_bldg[dist_col] <= threshold).astype('int8')

    # Calculate metrics for console printout
    n_flagged  = gdf_bldg[flag_col].sum()
    pct        = n_flagged / len(gdf_bldg) * 100
    binary_cols.append(flag_col)
    
    print(f'  {flag_col:<25} : {n_flagged:>10,}  ({pct:.1f}%)')

print(f'\nAll {len(binary_cols)} binary road features computed.')

# Road Density in 200m Buffer

ROAD_DENSITY_BUFFER = 200  # metres (EPSG:3035)

bldg_geoms_proj = bldg_proj.geometry.values
road_count_cols = []

print("Computing road density features (200m buffer)...")

for cat_name, cat_roads in roads_by_cat.items():
    col = f'road_count_{cat_name}_200m'
    
    if cat_roads.empty:
        gdf_bldg[col] = np.int16(0)
        road_count_cols.append(col)
        print(f"  {cat_name:<25}: EMPTY — filled with 0")
        continue

    tree  = STRtree(cat_roads.geometry.values)
    q_idx, _ = tree.query(bldg_geoms_proj, predicate='dwithin', distance=ROAD_DENSITY_BUFFER)
    counts   = np.bincount(q_idx, minlength=len(bldg_geoms_proj)).astype(np.int16)
    
    gdf_bldg[col] = counts
    road_count_cols.append(col)
    print(f"  {cat_name:<25}: {counts.sum():,} total road segments found")
    
    del tree, q_idx, counts
    gc.collect()

# ── Total road count across all types ────────────────────────────────
gdf_bldg['road_count_total_200m'] = (
    gdf_bldg[road_count_cols].sum(axis=1).astype(np.int16)
)
road_count_cols.append('road_count_total_200m')

# ── Proportion features (normalised by total — robust to urban density) ──
total = gdf_bldg['road_count_total_200m'].replace(0, np.nan)

proportion_map = {
    'pct_residential_roads_200m' : 'residential_road',
    'pct_rural_roads_200m'       : 'rural_track',
    'pct_service_roads_200m'     : 'service_road',
    'pct_major_roads_200m'       : 'major_road',
    'pct_secondary_roads_200m'   : 'secondary_road',
}

pct_cols = []
for new_col, cat_name in proportion_map.items():
    src_col = f'road_count_{cat_name}_200m'
    if src_col in gdf_bldg.columns:
        gdf_bldg[new_col] = (gdf_bldg[src_col] / total).fillna(0).astype(np.float32)
        pct_cols.append(new_col)

print(f"\nRoad density columns added  : {road_count_cols}")
print(f"Road proportion columns     : {pct_cols}")

# ── Quick sanity check ───────────────────────────────────────────────
print("\nMedian road_count_total_200m by building type:")
print(
    gdf_bldg.groupby('stage1_l1')['road_count_total_200m']
    .median().sort_values(ascending=False).round(1)
)

del roads_proj, roads_by_cat
gc.collect()
print("roads_proj freed.")


Applying Binary Thresholds...
  near_major_road_700m      : 13,830,920  (35.6%)
  near_secondary_road_115m  : 14,479,518  (37.3%)
  near_residential_road_25m : 19,883,519  (51.2%)
  near_service_road_35m     : 14,207,474  (36.6%)
  near_rural_track_250m     : 14,360,193  (37.0%)
  near_pedestrian_zone_25m  :     53,013  (0.1%)

All 6 binary road features computed.
Computing road density features (200m buffer)...
  residential_road         : 417,257,382 total road segments found
  secondary_road           : 108,078,675 total road segments found
  rural_track              : 28,092,811 total road segments found
  major_road               : 25,443,964 total road segments found
  service_road             : 449,427,832 total road segments found
  pedestrian_zone          : 2,050,434 total road segments found


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.


Road density columns added  : ['road_count_residential_road_200m', 'road_count_secondary_road_200m', 'road_count_rural_track_200m', 'road_count_major_road_200m', 'road_count_service_road_200m', 'road_count_pedestrian_zone_200m', 'road_count_total_200m']
Road proportion columns     : ['pct_residential_roads_200m', 'pct_rural_roads_200m', 'pct_service_roads_200m', 'pct_major_roads_200m', 'pct_secondary_roads_200m']

Median road_count_total_200m by building type:
stage1_l1
semi_commercial    52.0
commercial         36.0
transportation     34.0
civic              31.0
residential        27.0
filter             25.0
industrial         23.0
agricultural       10.0
military            7.0
Name: road_count_total_200m, dtype: float64
roads_proj freed.


In [ ]:
# buildings with no road features at all
all_zero = gdf_bldg[binary_cols].sum(axis=1) == 0
print(f'Buildings with all binary flags = 0: {all_zero.sum():,}')
print('These are buildings far from all road types — expected for very remote rural buildings.')
if 'area_sqm' in gdf_bldg.columns and all_zero.sum() > 0:
    print('\nArea distribution of zero-flag buildings:')
    print(gdf_bldg[all_zero]['area_sqm'].describe().round(1))

# Create the subset of isolated buildings
df_isolated = gdf_bldg[all_zero].copy()

# Bucket the areas to see the scale
df_isolated['size_category'] = pd.cut(
    df_isolated['area'], 
    bins=[0, 50, 200, 1000, np.inf], 
    labels=['Tiny (Shed/Cabin)', 'Medium (House)', 'Large (Barn/Warehouse)', 'Massive']
)

print("Isolated Buildings by Size:")
print(df_isolated['size_category'].value_counts())

# Check the building type 
if 'raw_label' in df_isolated.columns:
    print("\nTop OSM tags for isolated buildings:")
    print(df_isolated['raw_label'].value_counts().head(10))



Buildings with all binary flags = 0: 2,068,978
These are buildings far from all road types — expected for very remote rural buildings.
Isolated Buildings by Size:
size_category
Medium (House)            913597
Tiny (Shed/Cabin)         781047
Large (Barn/Warehouse)    321173
Massive                    53161
Name: count, dtype: int64

Top OSM tags for isolated buildings:
raw_label
house              112303
garage              86584
shed                50300
apartments          41292
detached            39163
allotment_house     36973
residential         36051
roof                13365
farm_auxiliary      12835
hut                 12012
Name: count, dtype: int64


In [25]:
gdf_waterways = gpd.read_parquet(WATERWAY_FILE)
gdf_railways  = gpd.read_parquet(RAILWAY_FILE)

print(f"  Waterways : {len(gdf_waterways):,}")
print(f"  Railways  : {len(gdf_railways):,}")
print(gdf_waterways['waterway_category'].value_counts())
print(gdf_railways['railway_category'].value_counts())



  Waterways : 1,557,870
  Railways  : 270,947
waterway_category
minor_waterway    1534402
major_waterway      23468
Name: count, dtype: int64
railway_category
heavy_rail    230412
light_rail     40535
Name: count, dtype: int64


In [ ]:
#  Waterway + Railway Distance & Density Features
print("Projecting waterways and railways to EPSG:3035...")
waterways_proj = gdf_waterways.to_crs(epsg=3035)
railways_proj  = gdf_railways.to_crs(epsg=3035)

MAX_DIST = 5000.0

waterway_railway_dist_cols   = []
waterway_railway_binary_cols = []
waterway_railway_count_cols  = []

# ── Thresholds (same logic as roads — based on physical meaning) ──────
# Heavy rail 500m: industrial buildings in Germany almost always within 500m of rail
# Major waterway 500m: manufacturing/utilities cluster beside navigable water
# Light rail 200m: urban signal — commercial/civic
# Minor waterway 300m: weaker signal but still useful
WR_THRESHOLDS = {
    'major_waterway' : 500,
    'minor_waterway' : 300,
    'heavy_rail'     : 500,
    'light_rail'     : 200,
}

# Group by category — same pattern as roads_by_cat
waterways_by_cat = {
    cat: waterways_proj[waterways_proj['waterway_category'] == cat].copy()
    for cat in waterways_proj['waterway_category'].dropna().unique()
}
railways_by_cat = {
    cat: railways_proj[railways_proj['railway_category'] == cat].copy()
    for cat in railways_proj['railway_category'].dropna().unique()
}

all_layers = {**waterways_by_cat, **railways_by_cat}

for cat_name, cat_gdf in all_layers.items():
    print(f"\n  Processing: {cat_name} ({len(cat_gdf):,} segments)...")

    dist_col  = f'dist_nearest_{cat_name}'
    threshold = WR_THRESHOLDS.get(cat_name, 500)
    flag_col  = f'near_{cat_name}_{threshold}m'
    count_col = f'count_{cat_name}_200m'

    if cat_gdf.empty:
        gdf_bldg[dist_col]  = np.float32(MAX_DIST)
        gdf_bldg[flag_col]  = np.int8(0)
        gdf_bldg[count_col] = np.int16(0)
        waterway_railway_dist_cols.append(dist_col)
        waterway_railway_binary_cols.append(flag_col)
        waterway_railway_count_cols.append(count_col)
        continue

    # ── Distance ──────────────────────────────────────────────────────
    dense      = cat_gdf.geometry.segmentize(max_segment_length=50)
    coords_df  = dense.get_coordinates()
    layer_coords = np.unique(
        np.column_stack([coords_df.x, coords_df.y]), axis=0
    )
    print(f"    {len(layer_coords):,} unique vertices in tree")

    tree = cKDTree(layer_coords)
    dist, _ = tree.query(bldg_coords, k=1, distance_upper_bound=MAX_DIST)
    gdf_bldg[dist_col] = np.where(dist == np.inf, MAX_DIST, dist).astype(np.float32)
    waterway_railway_dist_cols.append(dist_col)
    print(f"    {dist_col}: median={np.nanmedian(gdf_bldg[dist_col]):.0f}m")

    # ── Binary flag ───────────────────────────────────────────────────
    gdf_bldg[flag_col] = (gdf_bldg[dist_col] <= threshold).astype(np.int8)
    waterway_railway_binary_cols.append(flag_col)
    print(f"    {flag_col}: {gdf_bldg[flag_col].sum():,} buildings flagged")

    # ── 200m density count (same as road_count_X_200m) ────────────────
    str_tree    = STRtree(cat_gdf.geometry.values)
    q_idx, _    = str_tree.query(bldg_proj.geometry.values,
                                  predicate='dwithin', distance=200)
    counts      = np.bincount(q_idx, minlength=len(bldg_proj)).astype(np.int16)
    gdf_bldg[count_col] = counts
    waterway_railway_count_cols.append(count_col)
    print(f"    {count_col}: {counts.sum():,} total hits")

    del tree, str_tree, dense, coords_df, layer_coords, dist, q_idx, counts
    gc.collect()

del waterways_proj, railways_proj, waterways_by_cat, railways_by_cat, all_layers
gc.collect()

print(f"\nBlock C done.")
print(f"  Distance cols : {waterway_railway_dist_cols}")
print(f"  Binary cols   : {waterway_railway_binary_cols}")
print(f"  Count cols    : {waterway_railway_count_cols}")

Projecting waterways and railways to EPSG:3035...

  Processing: major_waterway (23,468 segments)...
    2,221,508 unique vertices in tree


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


    dist_nearest_major_waterway: median=1392m
    near_major_waterway_500m: 9,048,294 buildings flagged


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


    count_major_waterway_200m: 4,769,149 total hits

  Processing: minor_waterway (1,534,402 segments)...
    13,673,291 unique vertices in tree


...

    count_light_rail_200m: 7,076,180 total hits

  Processing: heavy_rail (230,412 segments)...
    2,634,759 unique vertices in tree


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


    dist_nearest_heavy_rail: median=1256m
    near_heavy_rail_500m: 10,011,850 buildings flagged


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


    count_heavy_rail_200m: 23,710,413 total hits

Block C done.
  Distance cols : ['dist_nearest_major_waterway', 'dist_nearest_minor_waterway', 'dist_nearest_light_rail', 'dist_nearest_heavy_rail']
  Binary cols   : ['near_major_waterway_500m', 'near_minor_waterway_300m', 'near_light_rail_200m', 'near_heavy_rail_500m']
  Count cols    : ['count_major_waterway_200m', 'count_minor_waterway_200m', 'count_light_rail_200m', 'count_heavy_rail_200m']


In [34]:
poi_ratio_cols = [
    c for c in gdf_bldg.columns
    if any(c.startswith(p) for p in [
        'food_share_', 'retail_share_', 'office_share_',
        'accom_share_', 'food_vs_retail_', 'office_vs_retail_',
        'comm_diversity_'
    ])
]

all_feature_cols = (
    geo_cols +
    ['dist_nearest_real_building', 'is_touching_real_building'] + 
    [f'dist_to_{z}' for z in ['residential', 'commercial', 'industrial']] +
    [f'poi_count_{cat}_{r}' for cat in NEEDS_DENSITY for r in ['25m', '50m', '100m', '250m']
     if f'poi_count_{cat}_{r}' in gdf_bldg.columns] +
    ['dist_nearest_transport', 'dist_nearest_education'] +
    ['poi_count_inside'] +
    dist_cols + binary_cols  + road_count_cols + pct_cols +
    waterway_railway_dist_cols + waterway_railway_binary_cols + waterway_railway_count_cols
    + zone_membership_cols + poi_ratio_cols
)

print(f"Total features : {len(all_feature_cols)}")
print(f"DataFrame shape: {gdf_bldg.shape}")

null_counts = gdf_bldg[all_feature_cols].isnull().sum()
if null_counts.any():
    print(f"\n⚠️  Columns with nulls:")
    print(null_counts[null_counts > 0])
else:
    print("\n✅ No nulls in any feature column.")

print("\nDescriptive stats:")
numeric_features = [c for c in all_feature_cols if gdf_bldg[c].dtype != object]
print(gdf_bldg[numeric_features].describe().round(2).to_string())

# Signal direction check against stage1 ground truth
if 'stage1_l1' in gdf_bldg.columns:
    print("\nSANITY: mean dist_to_residential by known label:")
    print(gdf_bldg.groupby('stage1_l1')['dist_to_residential'].median().round(1))
    print("\nSANITY: mean area by known label:")
    print(gdf_bldg.groupby('stage1_l1')['area'].median().round(1))

Total features : 107
DataFrame shape: (38802372, 156)



✅ No nulls in any feature column.

Descriptive stats:
              area    perimeter  compactness    convexity  num_vertices     mrr_long    mrr_short   elongation  rectangularity     diameter  dist_nearest_real_building  is_touching_real_building  dist_to_residential  dist_to_commercial  dist_to_industrial  poi_count_retail_25m  poi_count_retail_50m  poi_count_retail_100m  poi_count_retail_250m  poi_count_office_25m  poi_count_office_50m  poi_count_office_100m  poi_count_office_250m  poi_count_food_25m  poi_count_food_50m  poi_count_food_100m  poi_count_food_250m  poi_count_other_service_25m  poi_count_other_service_50m  poi_count_other_service_100m  poi_count_other_service_250m  poi_count_accommodation_25m  poi_count_accommodation_50m  poi_count_accommodation_100m  poi_count_accommodation_250m  poi_count_civic_25m  poi_count_civic_50m  poi_count_civic_100m  poi_count_civic_250m  poi_count_healthcare_25m  poi_count_healthcare_50m  poi_count_healthcare_100m  poi_count_healthcare_250m

In [35]:
print(f"Saving to {OUTPUT_FILE}...")
gdf_bldg.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved.")
print(f"   Shape  : {gdf_bldg.shape}")
print(f"   Columns: {gdf_bldg.columns.tolist()}")

Saving to /fast/home/o-olajuyigbe/osm_project/data/processed/germany_buildings_feature_engineered.parquet...
Saved.
   Shape  : (38802372, 156)
   Columns: ['id', 'geometry', 'building', 'tag_l1', 'tag_l2', 'is_abandoned', 'tag_is_mixed', 'tag_source', 'tag_used', 'all_candidates', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'name', 'amenity', 'building:use', 'craft', 'office', 'shop', 'tags', 'address_tier', 'compactness', 'building_l1', 'building_l2', 'building_is_mixed', 'buse_l1', 'buse_l2', 'buse_is_mixed', 'amenity_l1', 'amenity_l2', 'shop_l1', 'shop_l2', 'stage1_l1', 'stage1_l2', 'stage1_is_mixed', 'stage1_source', 'raw_label', 'stage2_l1', 'stage2_l2', 'stage2_source', 'landuse_l1', 'landuse_l2', 'landuse_used', 'landuse_osm_id', 'has_building_levels', 'building_levels', 'has_name', 'area', 'perimeter', 'convexity', 'num_vertices', 'mrr_long', 'mrr_short', 'elongation', 'rectangularity', 'diameter', 'dist_nearest_real_building', 'is_touching_real_building',